In [6]:
import pandas as pd

interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

interactions.head(10)

,user_idx,movie_idx,rating,timestamp
0,0,1104,5.0,978300760
1,0,639,3.0,978302109
2,0,853,3.0,978301968
3,0,3177,4.0,978300275
4,0,2162,5.0,978824291
5,0,1107,3.0,978302268
6,0,1195,5.0,978302039
7,0,2599,5.0,978300719
8,0,580,4.0,978302268
9,0,858,4.0,978301368


In [7]:
def recommend_popular(
        user_idx: int,
        interactions: pd.DataFrame,
        k: int = 10
):
    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            rating_count="count",
            avg_rating="mean"
        )
        .reset_index()
        .sort_values(by=["rating_count", "avg_rating"], ascending=False)
    )

    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .head(k)
    )

print(recommend_popular(user_idx=3, interactions=interactions, k=10))


      movie_idx  rating_count  avg_rating
2651       2651          3428    4.317386
575         575          2649    4.058513
2374       2374          2590    4.315830
1178       1178          2583    3.990321
579         579          2578    4.351823
1449       1449          2538    3.739953
593         593          2513    4.254676
2557       2557          2459    4.406263
106         106          2443    4.234957
2203       2203          2369    4.127480


In [8]:
def bayesian_popularity(
    interactions: pd.DataFrame,
    m: int = 100
):
    C = interactions.rating.mean()

    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            rating_count="count",
            avg_rating="mean"
        )
        .reset_index()
    )

    popularity["score"] = (
        popularity.rating_count 
        / (popularity.rating_count + m) 
        * popularity.avg_rating 
        + m 
        / (popularity.rating_count + m)
        * C
    )

    return popularity.sort_values(by="score", ascending=False)

def recommend_bayesian_popularity(
    user_idx: int,
    popularity: pd.DataFrame,
    interactions: pd.DataFrame,
    k: int = 10
):
    
    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .reset_index()
        .head(k)
        .sort_values(by="score", ascending=False)
        .movie_idx
    )


model = bayesian_popularity(interactions=interactions, m=100)

print(recommend_bayesian_popularity(1, model, interactions, k=10))

0     802
1     513
2      49
3    1839
4     253
5    1066
6     843
7     708
8     713
9    2557
Name: movie_idx, dtype: int64


In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit

train_parts, val_parts, test_parts = TemporalSplit().split(data=interactions)

print(train_parts.shape)
print(test_parts.shape)
print(val_parts.shape)

(797758, 4)
(105732, 4)
(96719, 4)


In [10]:
import sys
from pathlib import Path
import numpy as np

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)

recalls = []
precisions = []
ndcgs = []

k = 10

for user_idx in interactions.user_idx.unique():

    recommended = recommend_bayesian_popularity(user_idx, model, train_parts).tolist()
    test_user = test_parts[test_parts.user_idx == user_idx]

    relevant = test_user.loc[
        test_user.rating >= 4,
        "movie_idx"
    ].tolist()

    if len(relevant) == 0:
        continue

    recalls.append(recall_at_k(relevant, recommended, k))
    precisions.append(precision_at_k(relevant, recommended, k))
    ndcgs.append(ndcg_at_k(relevant, recommended, k))

recalls = np.array(recalls)
precisions = np.array(precisions)
ndcgs = np.array(ndcgs)

print(
    f"recall at k: {recalls.mean()}\n"
    f"precision at k: {precisions.mean()}\n"
    f"ndcg at k: {ndcgs.mean()}"
)

recall at k: 0.026966619206610483
precision at k: 0.026078063746378048
ndcg at k: 0.03278680179439576
